## `Материалы кафедры ММП факультета ВМК МГУ. Введение в эффективные системы глубокого обучения.`

## `Задание 04. Работа с большими моделями`

#### Фамилия, имя:

Дата выдачи: <span style="color:red">__04.11.2026__</span>.

Мягкий дедлайн: <span style="color:red">__18.11.2026 15:00__</span>.

Стоимость: __10 баллов__ (основная часть заданий) + __3 балла__ (дополнительные задания).

<span style="color:red">__В ноутбуке все клетки должны выполняться без ошибок при последовательном их выполнении.__</span>

#### `Москва, 2026`

Авторы задания: `@sir_rois` и `@jserdyuk`

## `Постановка задачи`

В данном задании вы научитесь обучать модель при помощи LoRA, а также реализуете CPU offloading.

Используемая в задании модель Qwen/Qwen2.5-0.5B

Используемый датасет - GSM8K (набор несложных математических задач с решением и ответом)

## `Необходимая теория`

### `LoRA (Low-Rank Adaptation)`
**LoRA** — это метод адаптации больших языковых моделей, который позволяет обучать только небольшие низкоранговые матрицы вместо всех весов модели.  
- Позволяет **экономить память и время обучения**.  
- Подходит для **тонкой настройки (fine-tuning)** моделей без переобучения всех параметров.  
- Обучение ограничивается несколькими адаптационными матрицами, а основная модель остаётся неизменной.

Идея простая: вместо того чтобы обновлять все веса слоя `W`, мы добавляем **низкоранговую поправку**:

$$
W' = W + \Delta W, \quad \Delta W = B A
$$

где:  
- $A \in \mathbb{R}^{r \times d_\text{in}}$ и $B \in \mathbb{R}^{d_\text{out} \times r}$ — **малые матрицы для адаптации**,  
- $r \ll d_\text{in}, d_\text{out}$ — низкий ранг,  
- дополнительно можно использовать **масштабирование** и **dropout** для стабилизации обучения.



### `CPU Offloading`
**CPU Offloading** — это техника, когда часть модели (например, слои или параметры) хранится и обрабатывается на **CPU**, а не на GPU.  
- Позволяет работать с **очень большими моделями**, которые не помещаются целиком на GPU.  
- GPU используется только для активного вычисления, что снижает требования к памяти GPU.

### `CPU Offloading with Overlap`

**CPU Offloading with Overlap** — это усовершенствованная техника CPU Offloading, при которой **перемещение слоев между CPU и GPU происходит параллельно с вычислениями на GPU**.  

- **Обычный CPU Offloading:** слой загружается на GPU → выполняется вычисление → выгружается обратно на CPU.  
  Это может создавать паузы, потому что GPU ждёт загрузки слоя.  

- **С Overlap:**  
  - Пока GPU вычисляет текущий слой, **следующий слой уже подгружается с CPU**.  
  - Это **сокращает простой GPU**, повышая скорость обработки.  

Идея в том, чтобы **сглаживать задержки при работе с большими моделями**, которые не помещаются целиком на GPU.

<span style="color:red">ВНИМАНИЕ!</span>.
 Ячейки, в которых ожидается ваш письменный ответ, помечены (❓). Перед отправкой ноутбука проверьте, что вы ответили на все вопросы 

**Как оформлять решение.** Все пункты заданий, включая подготовку данных,
оценку, оборачивание слоёв и циклы обучения, нужно реализовать самостоятельно.
Повторяющийся код оформляйте в свои функции и используйте их в следующих частях:
например, оборачивание слоёв и один шаг обучения достаточно написать один раз.
Копировать `forward` всей Qwen для оборачивания её блоков не требуется.
Ограничения на число строк нет.

Ниже добавлены открытые проверки. Они помогают найти типичные ошибки, но не
заменяют эксперименты из условий. Тесты токенизации и LoRA работают на CPU;
проверки offloading требуют CUDA и явно сообщают, если она недоступна.
Для сравнения обучения используйте одинаковые начальные веса, одинаковые батчи,
`use_cache=False` и отключённый dropout. При реализации checkpointing явно
укажите `use_reentrant` и учитывайте поведение выбранного варианта.

## `Часть 1. Обучение LoRA (2 балла)`

Начнем с импорта необходимых библиотек, загрузки модели, разбиения датасета на трейн, валидацию и тест.

In [ ]:
import re
from tqdm import tqdm
import torch
from torch import nn
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from datasets import load_dataset
import torch.nn.functional as F

model_id = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    trust_remote_code=True
)

dataset = load_dataset("gsm8k", "main", split="train")

train_eval_test = dataset.train_test_split(test_size=0.1, seed=42)
eval_test_dataset = train_eval_test['test']
train_dataset = train_eval_test['train']

eval_test_split = eval_test_dataset.train_test_split(test_size=0.5, seed=42)
test_dataset = eval_test_split['train']
eval_dataset = eval_test_split['test']

**Задание 1 (2 балла)**. Реализация LoRA. В этом задании необходимо дообучить LoRA и повысить качество после решения задач. Для выполнения данного задания нужно реализовать следующие пункты: 
1. Посмотрите на данные, которые хранятся в датасете и токенизируйте. Не забудьте, что для обучения модели лучше не использовать токены, соответствующие условию.
1. Реализуйте функцию evaluate_generation, которая замеряет качество модели на тестовом датасете. В качестве метрики качества используйте `accuracy` предсказаний. Используйте few-shot с 3мя примерами решений для того, чтобы модель поняла формат, в котором вы ожидаете ответ.
1. Реализуйте класс `LoRALinear`, который оборачивает модули модели, добавляя обучаемые матрицы для получения предсказаний. Заморозьте модель, оберните модули внимания (`q_proj`, `k_proj`, `v_proj`, `o_proj`)
1. Замерьте качество модели до обучения, обучите модель, замерьте качество после обучения.
1. Сделайте выводы

В `labels` маскируйте условие и padding значением `-100`, сохраняя хотя бы один
токен решения. Сдвиг labels вручную не нужен: его выполняет causal LM.
При инициализации LoRA поправка должна быть нулевой; число обучаемых параметров
одного слоя равно `r * (in_features + out_features)`.

In [ ]:
def tokenize(sample: dict) -> dict:
    """
    Convert a single dataset example into the desired format.

    Args:
        sample (dict): One example containing fields like 'question' and 'answer'.

    Returns:
        dict: A dictionary with the following keys:
            - 'input_ids' (list): Token IDs of the input text.
            - 'attention_mask' (list): Attention mask (1 for tokens to attend to, 0 to ignore).
            - 'labels' (list): Token IDs used for loss computation (can contain -100 for tokens to ignore).
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Реализуйте tokenize")

In [ ]:
def test_tokenize():
    samples = [
        {"question": "What is 2 + 3?", "answer": "2 + 3 = 5.\n#### 5"},
        {"question": "What is 2 - 5?", "answer": "2 - 5 = -3.\n#### -3"},
    ]
    for sample in samples:
        result = tokenize(sample)
        ids = torch.as_tensor(result["input_ids"])
        mask = torch.as_tensor(result["attention_mask"])
        labels = torch.as_tensor(result["labels"])
        assert ids.ndim == mask.ndim == labels.ndim == 1, "Ожидается один пример без batch dimension"
        assert len(ids) == len(mask) == len(labels) > 0, "Поля должны быть непустыми и одной длины"
        assert not ids.is_floating_point() and not labels.is_floating_point(), "Token IDs должны быть целыми"
        assert ((mask == 0) | (mask == 1)).all(), "attention_mask должна содержать 0/1"
        assert ((ids >= 0) & (ids < len(tokenizer))).all(), "Недопустимый token ID"
        active = labels != -100
        assert active.any(), "Все labels замаскированы: loss не по чему считать"
        assert ((labels == -100) & (mask == 1)).any(), "Не замаскировано условие"
        assert (labels[mask == 0] == -100).all(), "Padding должен быть исключён из loss"
        assert torch.equal(labels[active], ids[active]), "Не сдвигайте labels вручную"
        assert labels[mask.bool()][0] == -100, "Начало условия должно быть замаскировано"
    print("tokenize: OK (проверьте границу условия и решения также вручную)")

test_tokenize()

In [ ]:
train_dataset = train_dataset.map(tokenize)
eval_dataset = eval_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

In [ ]:
from typing import Union
from transformers import PreTrainedModel, PreTrainedTokenizerBase
from datasets import Dataset

def evaluate_generation(
    model: PreTrainedModel,
    dataset: Dataset,
    tokenizer: PreTrainedTokenizerBase,
    batch_size: int = 4,
    few_shot_size: int = 3
) -> float:
    """
    Evaluate the model's generation accuracy on a dataset.

    Args:
        model (PreTrainedModel): The language model to evaluate.
        dataset (Dataset): The dataset containing examples to generate predictions for.
        tokenizer (PreTrainedTokenizerBase): Tokenizer corresponding to the model.
        batch_size (int, optional): Number of examples per batch. Defaults to 4.
        few_shot_size (int, optional): Number of few-shot examples to prepend. Defaults to 3.

    Returns:
        float: Accuracy of the model's predictions on the dataset.
    """
    ### YOUR CODE HERE
    raise NotImplementedError("Реализуйте evaluate_generation")

In [ ]:
import math

class LoRALinear(nn.Module):
    def __init__(self, module, r=8, alpha=16, dropout=0.05):
        super().__init__()
        ### YOUR CODE HERE
        self.module = module

    def forward(self, x):
        ### YOUR CODE HERE
        return self.module(x)

In [ ]:
def test_lora():
    with torch.random.fork_rng():
        torch.manual_seed(42)
        for rank in (1, 3):
            base = nn.Linear(7, 5)
            base.requires_grad_(False)
            saved_base = {name: p.detach().clone() for name, p in base.named_parameters()}
            layer = LoRALinear(base, r=rank, alpha=2 * rank, dropout=0.0)
            trainable = [p for p in layer.parameters() if p.requires_grad]
            assert sum(p.numel() for p in trainable) == rank * (7 + 5), "Неверное число параметров LoRA"
            assert not any(p.requires_grad for p in base.parameters()), "Базовый слой должен остаться замороженным"
            for shape in ((4, 7), (2, 3, 7)):
                x = torch.randn(shape)
                torch.testing.assert_close(layer(x), base(x), msg="При инициализации LoRA должна сохранять выход слоя")
            x, target = torch.randn(2, 3, 7), torch.randn(2, 3, 5)
            initial = layer(x).detach().clone()
            optimizer = torch.optim.SGD(trainable, lr=0.1)
            for _ in range(3):
                optimizer.zero_grad(set_to_none=True)
                F.mse_loss(layer(x), target).backward()
                assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in trainable), "Потерян градиент адаптера"
                optimizer.step()
            assert not torch.allclose(layer(x), initial), "LoRA не обучается (проверьте инициализацию обеих матриц)"
            for name, p in base.named_parameters():
                torch.testing.assert_close(p, saved_base[name], rtol=0, atol=0, msg=f"Изменился базовый параметр {name}")
                assert p.grad is None, f"Градиент замороженного параметра {name}"
    print("LoRA: OK")

test_lora()

In [ ]:
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

for param in model.parameters():
    param.requires_grad = False

for name, module in model.named_modules():
    pass
    # Замените все слои, описанные в target_modules на LoRALinear
    ### YOUR CODE HERE


total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

assert total_params == 495114112
assert trainable_params == 1081344

print(f"Percentage of trainable params: {100 * trainable_params / total_params:.2f}%")

In [ ]:
print("Генеративная оценка до обучения:")
evaluate_generation(model, test_dataset, tokenizer)

In [ ]:
args = TrainingArguments(
    ### YOUR CODE HERE
)

trainer = Trainer(
    ### YOUR CODE HERE
)

In [ ]:
batch = trainer.data_collator([train_dataset[i] for i in range(2)])
assert batch["input_ids"].shape == batch["labels"].shape == batch["attention_mask"].shape
for i in range(2):
    attended = batch["attention_mask"][i].bool()
    original = train_dataset[i]
    original_mask = torch.as_tensor(original["attention_mask"]).bool()
    expected_labels = torch.as_tensor(original["labels"])[original_mask]
    assert torch.equal(batch["labels"][i][attended].cpu(), expected_labels), "Collator изменил labels; проверьте маску условия"
assert (batch["labels"][batch["attention_mask"] == 0] == -100).all(), "Padding в батче должен быть замаскирован"
print("Collator: OK")

In [ ]:
trainer.train()

In [ ]:
print("Генеративная оценка после обучения:")
evaluate_generation(model, test_dataset, tokenizer)

**Выводы** (❓):

...

## `Часть 2. Реализация инференса с CPU Offloading (1 балл)`

Вновь начнем с загрузки модели и подготовки референсного предсказания

In [ ]:
import re
from tqdm import tqdm
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from datasets import load_dataset
import torch.nn.functional as F

model_id = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
input_ids = tokenizer.encode("Hello, world!", return_tensors="pt")
config = transformers.AutoConfig.from_pretrained(model_id, trust_remote_code=True)


model.eval()
with torch.no_grad():
    res_baseline = model(input_ids=input_ids, use_cache=False)

Воспользуемся функией проверки того, что все слои на cpu

In [ ]:
import torch
from torch import nn

def test_model_on_cpu(model: nn.Module) -> bool:
    """
    Check that all layers in `model.model.layers` are on CPU.

    Args:
        model (nn.Module): The model to check, expected to have `model.model.layers`.

    Returns:
        bool: True if all layers are on CPU, False otherwise.
    """
    for i, layer in enumerate(model.model.layers):
        for param in list(layer.parameters()) + list(layer.buffers()):
            if param.device.type != 'cpu':
                return False
    return True

test_model_on_cpu(model)

In [ ]:
def check_block(wrapper_cls, *, backward=False):
    import copy

    if not torch.cuda.is_available():
        raise RuntimeError("Для теста offloading нужна CUDA; это не успешное прохождение теста")

    class TinyBlock(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear = nn.Linear(7, 5)
            self.register_buffer("offset", torch.ones(5))
            self.calls = 0

        def forward(self, x, *, scale=1.0):
            self.calls += 1
            assert self.linear.weight.is_cuda and self.offset.is_cuda, "Блок должен вычисляться на GPU"
            return (torch.tanh(self.linear(x) + self.offset) * scale,)

    def on_cpu(module):
        tensors = list(module.parameters()) + list(module.buffers())
        assert all(t.device.type == "cpu" for t in tensors), "Блок не вернулся на CPU"

    def output_tensor(output):
        return output[0] if isinstance(output, tuple) else output

    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(42)
        original = TinyBlock()
        reference = copy.deepcopy(original).cuda()
        parameters = list(original.named_parameters())
        wrapped = wrapper_cls(original, device="cuda")
        assert {id(p) for p in wrapped.parameters()} == {id(p) for _, p in parameters}, "Обёртка должна сохранять исходные Parameter"
        for shape in ((2, 7), (2, 3, 7)):
            reference.zero_grad(set_to_none=True)
            wrapped.zero_grad(set_to_none=True)
            x_ref = torch.randn(shape, device="cuda", requires_grad=backward)
            x_test = x_ref.detach().clone().requires_grad_(backward)
            calls_before = original.calls
            with torch.set_grad_enabled(backward):
                expected = output_tensor(reference(x_ref, scale=0.7))
                actual = output_tensor(wrapped(x_test, scale=0.7))
                assert actual.device == expected.device, "Выход блока должен остаться на GPU"
                torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-5)
            on_cpu(original)
            if backward:
                expected.square().mean().backward()
                actual.square().mean().backward()
                assert x_test.grad is not None, "Потерян градиент по входу"
                torch.testing.assert_close(x_test.grad, x_ref.grad, rtol=1e-4, atol=1e-5)
                for (name, p), p_ref in zip(parameters, reference.parameters()):
                    assert p.grad is not None, f"Потерян градиент {name}"
                    torch.testing.assert_close(p.grad.cpu(), p_ref.grad.cpu(), rtol=1e-4, atol=1e-5, msg=f"Градиент {name}")
                assert original.calls - calls_before >= 2, "При backward не было повторного вычисления блока"
                on_cpu(original)
    print(f"{wrapper_cls.__name__}: OK (два вызова, kwargs, параметры и buffers" + (", backward)" if backward else ")"))

**Задание 2 (1 балл)**. Реализация `CPU Offloading`. Получить инференс модели про помощи простого CPU offloading, в ходе которого необходимый слой грузится с CPU на GPU, получается его предсказание и выгружается обратно. Для выполнения данного задания нужно реализовать следующие пункты: 
1. Реализуйте класс `OffloadBlock`, который оборачивает модули модели и модифицирует их `forward`. Оберните слои трансформера. Слои с нормализацией, эмбедингами и lm_head всегда держите на GPU.
1. Получите инференс, сравните с референсными значениями

Обёртка должна сохранять объекты параметров исходного блока, передавать его
`args`/`kwargs` и переносить вместе с параметрами buffers. Выход остаётся на GPU.

In [ ]:
class OffloadBlock(nn.Module):
    def __init__(self, module: nn.Module, device:str = 'cuda'):
        ### YOUR CODE HERE
        pass

    def forward(self, *args, **kwargs):
        ### YOUR CODE HERE
        # Загружаем на GPU
        # Вызываем forward модуля
        out = ...
        # Выгружаем на CPU
        return out

In [ ]:
check_block(OffloadBlock)

In [ ]:
# Оберните слои трансформера. Слои с нормализацией, эмбедингами и lm_head всегда держите на gpu.
### YOUR CODE HERE

In [ ]:
model.eval()
with torch.no_grad():
    res_offloading = model(input_ids=input_ids.to('cuda'), use_cache=False)

torch.testing.assert_close(res_baseline.logits, res_offloading.logits.cpu(), rtol=1e-4, atol=1e-5)
assert test_model_on_cpu(model), "После forward decoder blocks должны быть на CPU"

## `Часть 3. Реализация обучения с CPU Offloading + checkpointing (1 балл)`

Вновь начнем с загрузки модели

In [ ]:
import re
from tqdm import tqdm
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from datasets import load_dataset
import torch.nn.functional as F

model_id = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
input_ids = tokenizer.encode("Hello, world!", return_tensors="pt")
config = transformers.AutoConfig.from_pretrained(model_id, trust_remote_code=True)

Подготовим функцию, которая берет две модели и сравнивает их градиенты

In [ ]:
# Если на этом этапе выпадает OOM - можете для дебага уменьшить число слоёв модели.
# config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
# config.num_hidden_layers = 5
# model = Qwen2ForCausalLM(config)


def compare_grads(model1, model2):
    params1, params2 = list(model1.named_parameters()), list(model2.named_parameters())
    assert len(params1) == len(params2) > 0, "Параметры потеряны или добавлены лишние копии"
    for (n1, p1), (n2, p2) in zip(params1, params2):
        label = f"{n1} <-> {n2}"
        assert p1.shape == p2.shape and p1.requires_grad == p2.requires_grad, label
        torch.testing.assert_close(p1.detach().cpu(), p2.detach().cpu(), rtol=1e-4, atol=1e-5, msg=f"Веса: {label}")
        if p1.requires_grad:
            assert p1.grad is not None and p2.grad is not None, f"Нет градиента: {label}"
            assert torch.isfinite(p1.grad).all() and torch.isfinite(p2.grad).all(), f"NaN/Inf: {label}"
            torch.testing.assert_close(p1.grad.cpu(), p2.grad.cpu(), rtol=1e-4, atol=1e-5, msg=f"Градиенты: {label}")
    print("Веса и градиенты: OK")

**Задание 3 (1 балл)**. Реализация `CPU Offloading` + `Checkpointing`. Обучить модель при помощи CPU offloading и checkpointing, в ходе которого модели модели будут двигаться на gpu и обратно, а промежуточные результаты вычисляться заново при помощи ванильной реализации `checkpointing` из PyTorch. Для выполнения данного задания нужно реализовать следующие пункты: 
1. Реализуйте класс `OffloadBlockWithCheckpointing`, который оборачивает модули модели и модифицирует их `forward`. В ходе `forward` нужно получить предсказания модуля с помощью `checkpoint`. Оберните слои трансформера. Слои с нормализацией, эмбедингами и lm_head всегда держите на GPU.
1. Возьмите также модель без оффлоадинга.
1. Совершите по три шага обучения для обеих моделей на идентичных входных сэмплах.
1. При помощи функции `compare_grads` проверьте, что градиенты после трех шагов совпадают.


Используйте две копии одних начальных весов и одинаковые батчи. Проверяйте
`compare_grads` после `backward` каждого шага (до `optimizer.step`), а также
после последнего обновления весов, до обнуления градиентов. Для точного сравнения
отключите dropout. Исходные параметры должны сохранять порядок регистрации.
После полного forward и backward decoder blocks должны вернуться на CPU.

In [ ]:
from torch.utils.checkpoint import checkpoint

class OffloadBlockWithCheckpointing(nn.Module):
    def __init__(self, module: nn.Module, layer=-1, device='cuda'):
        ### YOUR CODE HERE
        pass

    def forward(self, *args, **kwargs):
        ### YOUR CODE HERE
        return out

In [ ]:
check_block(OffloadBlockWithCheckpointing, backward=True)

In [ ]:
# Оберните слои трансформера. Слои с нормализацией, эмбедингами и lm_head всегда держите на gpu.
# Оформите это в свою функцию и используйте её в следующих частях.
### YOUR CODE HERE

In [ ]:
# Совершите по три шага обучения для обеих моделей на идентичных входных сэмплах.
# Оформите один шаг в свою функцию train_step и переиспользуйте её далее.
### YOUR CODE HERE

In [ ]:
compare_grads(model_wo_offloading, model_w_offloading)
assert test_model_on_cpu(model_w_offloading), "После backward decoder blocks должны быть на CPU"

## `Часть 4. Реализация обучения при помощи torch.autograd.Function (2 балла)`

Вновь начнем с загрузки модели

In [ ]:
import re
from tqdm import tqdm
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from datasets import load_dataset
import torch.nn.functional as F

model_id = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
input_ids = tokenizer.encode("Hello, world!", return_tensors="pt")
config = transformers.AutoConfig.from_pretrained(model_id, trust_remote_code=True)

Подготовим функцию, которая берет две модели и сравнивает их градиенты

In [ ]:
# Используйте compare_grads из части 3

**Задание 4 (2 балла)**. Реализация обучения при помощи `torch.autograd.Function`. Обучить модель, реализовав методы `offloading` и `checkpointing` внутри методов функции. Для выполнения данного задания нужно реализовать следующие пункты: 
1. Реализуйте класс `OffloadFunction`, внутри которого содержатся реализации функций `forward` и `backward`. Внутри этих функция должен быть реализован `offloading` и `checkpointing`.
1. Реализуйте класс `OffloadBlockWithFunction`, который оборачивает модули модели и модифицирует их `forward`, применяя функцию `OffloadFunction`. Повторное вычисление блока реализуйте самостоятельно внутри `backward`. В данном задани пользоваться реализацией `checkpoint` из PyTorch <span style="color:red">запрещено</span>. Оберните слои трансформера. Слои с нормализацией, эмбедингами и lm_head всегда держите на GPU.
1. Возьмите также модель без оффлоадинга.
1. Совершите по три шага обучения для обеих моделей на идентичных входных сэмплах.
1. При помощи функции compare_grads проверьте, что градиенты после трех шагов совпадают.


Используйте две копии одних начальных весов и одинаковые батчи. Проверяйте
`compare_grads` после `backward` каждого шага (до `optimizer.step`), а также
после последнего обновления весов, до обнуления градиентов. Для точного сравнения
отключите dropout. Исходные параметры должны сохранять порядок регистрации.
После полного forward и backward decoder blocks должны вернуться на CPU.

In [ ]:
class OffloadFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, *args):
        ### YOUR CODE HERE
        raise NotImplementedError("Реализуйте forward")

    @staticmethod
    def backward(ctx, *grad_outputs):
        ### YOUR CODE HERE
        raise NotImplementedError("Реализуйте backward")

In [ ]:
class OffloadBlockWithFunction(nn.Module):
    def __init__(self, module: nn.Module, device='cuda'):
        super().__init__()
        ### YOUR CODE HERE

    
    def forward(self, *args, **kwargs):
        ### YOUR CODE HERE
        pass

In [ ]:
check_block(OffloadBlockWithFunction, backward=True)

In [ ]:
# Оберните слои трансформера. Слои с нормализацией, эмбедингами и lm_head всегда держите на gpu.
# Переиспользуйте свою функцию оборачивания; при необходимости расширьте её.
### YOUR CODE HERE

In [ ]:
# Совершите по три шага обучения для обеих моделей на идентичных входных сэмплах.
# Переиспользуйте свой train_step из части 3.
### YOUR CODE HERE

In [ ]:
compare_grads(model_wo_offloading, model_w_offloading)
assert test_model_on_cpu(model_w_offloading), "После backward decoder blocks должны быть на CPU"

## `Часть 5. Реализация обучения с оффлоадингом и оверлапом (4 балла)`

Ну вы поняли

In [ ]:
import re
from tqdm import tqdm
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from datasets import load_dataset
import torch.nn.functional as F

model_id = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
input_ids = tokenizer.encode("Hello, world!", return_tensors="pt")
config = transformers.AutoConfig.from_pretrained(model_id, trust_remote_code=True)

In [ ]:
# Используйте compare_grads из части 3

**Задание 5 (3.5 балла)** Реализуйте механизм асинхронного оффлоадинга для модели Qwen2.5. Во время вычислений текущего слоя модель должна заранее подгружать следующий слой и асинхронно выгружать предыдущий, используя несколько CUDA Stream и CUDA Events для синхронизации. Для повышения эффективности необходимо реализовать буфер из трёх слоёв и использовать pinned memory при передаче данных между CPU и GPU.


1. Реализуйте класс `OffloadFunctionWithOverlap`, наследующий `torch.autograd.Function`.
   В нём должны быть реализованы методы `forward` и `backward`, обеспечивающие:
   1. Асинхронную загрузку следующего слоя (prefetch) во время вычислений текущего;
   1. Асинхронную выгрузку предыдущего слоя (offload);
   1. Синхронизацию потоков при помощи `torch.cuda.Event` и `wait_event`;
   1. Использование pinned memory для всех данных, передаваемых между CPU и GPU;
   1. Корректное сохранение необходимых тензоров для `backward`.
<br><br>
1. Класс `OffloadBlockWithOverlapFunction`, оборачивающий блоки трансформера.
   Он должен:
   1. Инициализировать два CUDA Streams — `prefetch_stream` и `offload_stream`;
   1. При каждом шаге:
      * Выполнять вычисления текущего слоя;
      * Загружать следующий слой на GPU в `prefetch_stream`;
      * Выгружать предыдущий слой обратно на CPU в `offload_stream`;
   1. Корректно управлять жизненным циклом потоков и событий (`event.record`, `wait_event`);
   1. Удерживать слои embedding, norm и lm_head постоянно на GPU
<br><br>
1. Возьмите также модель без оффлоадинга.
1. Совершите по три шага обучения для обеих моделей на идентичных входных сэмплах.
1. При помощи функции compare_grads проверьте, что градиенты после трех шагов совпадают.


**Практические рекомендации**
1. Можете воспользоваться `torch.profiler` или `nsys` для проверки, что операции `memcpyHtoD` и `kernel` выполняются параллельно.
1. Убедитесь, что pinned memory действительно используется: без неё non_blocking=True не даст эффекта.
1. Не выгружайте слой, пока его тензоры ещё участвуют в вычислениях.
1. Для стабильности можно добавить мягкую синхронизацию: `wait_stream(prefetch_stream)` перед завершением итерации.


Используйте две копии одних начальных весов и одинаковые батчи. Проверяйте
`compare_grads` после `backward` каждого шага (до `optimizer.step`), а также
после последнего обновления весов, до обнуления градиентов. Для точного сравнения
отключите dropout. Исходные параметры должны сохранять порядок регистрации.

In [ ]:
class OffloadFunctionWithOverlap(torch.autograd.Function):
    @staticmethod
    def forward(ctx, *args):
        # Дополните сигнатуру необходимыми аргументами.
        ### YOUR CODE HERE
        raise NotImplementedError("Реализуйте forward")

    @staticmethod
    def backward(ctx, *grad_outputs):
        ### YOUR CODE HERE
        raise NotImplementedError("Реализуйте backward")

In [ ]:
class OffloadBlockWithOverlapFunction(nn.Module):
    def __init__(self, module: nn.Module, device='cuda'):
        super().__init__()
        ### YOUR CODE HERE

    
    def forward(self, *args, **kwargs):
        ### YOUR CODE HERE
        pass

In [ ]:
# Оберните слои трансформера. Слои с нормализацией, эмбедингами и lm_head всегда держите на gpu.
# Переиспользуйте свою функцию оборачивания; при необходимости расширьте её.
### YOUR CODE HERE

In [ ]:
# Совершите по три шага обучения для обеих моделей на идентичных входных сэмплах.
# Переиспользуйте свой train_step из части 3.
### YOUR CODE HERE

In [ ]:
compare_grads(model_wo_offloading, model_w_offloading)

In [ ]:
# Дополнительная проверка переиспользования состояния overlap после обучения.
def test_repeated_inference(reference, candidate):
    modes = reference.training, candidate.training
    reference.eval()
    candidate.eval()
    try:
        with torch.random.fork_rng(), torch.no_grad():
            torch.manual_seed(42)
            for batch_size, length in ((1, 4), (2, 9), (1, 4)):
                ids = torch.randint(0, reference.config.vocab_size, (batch_size, length))
                expected = reference(input_ids=ids.to(reference.get_input_embeddings().weight.device), use_cache=False).logits
                actual = candidate(input_ids=ids.to(candidate.get_input_embeddings().weight.device), use_cache=False).logits
                torch.testing.assert_close(actual.cpu(), expected.cpu(), rtol=1e-4, atol=1e-5)
                resident = sum(any(p.is_cuda for p in layer.parameters()) for layer in candidate.model.layers)
                assert resident <= 3, "На GPU осталось больше трёх decoder blocks"
    finally:
        reference.train(modes[0])
        candidate.train(modes[1])
    print("Повторные вызовы модели: OK")

test_repeated_inference(model_wo_offloading, model_w_offloading)

**Задание 6 (0.5 балла)**

Приложите скриншот результатов работы профилировщика, на котором виден overlap загрузки и вычислений

Совпадение выходов и градиентов не доказывает наличие overlap. На скриншоте должны
быть видны одновременно выполняющиеся переносы и вычисления. Ускорение на
фиксированный процент не требуется: оно зависит от оборудования и размеров задачи.

## `Часть 6. Бонусы (3 балла)`

1. (1 балл). Покажите, что написанные вами функции и классы действительно помогают работать с большими моделями. Для этого покажите, что вы умеете работать с `Qwen2.5-7B` и потребляете меньше 10 ГБ видеопамяти, при этом GPU работает.
2. (1 балл). Составьте сводную таблицу, сколько времени занимает forward и backward, а также сколько памяти требует каждый из реализованных вами методов. В качестве бейзлайнов возьмите полностью GPU (если возможно) и полностью CPU реализации.
3. (1 балл). Реализуйте offloading состояния оптимизатора. Проверьте, что использование такой версии оптимизатора приводит к тому, что обучение совпадает с обычной версией.